# Import Necessary Packages

In [ ]:
import os
import string
import re
import nltk
import pandas as pd
import seaborn as sns
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from textblob import Word
from unidecode import unidecode
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MaxAbsScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from textblob import TextBlob

In [ ]:
tqdm.pandas()

In [ ]:
# setting LOKY_MAX_CPU_COUNT to the number of cores you want to use
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

# Loading the data

In [ ]:
train_df = pd.read_csv(os.path.join(".","data","training_data.csv"))
test_df = pd.read_csv(os.path.join(".","data","testing_data.csv"))

In [ ]:
train_df.head(3)

# EDA Checklist

In [ ]:
print(f'Training shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

In [ ]:
train_df.describe()

In [ ]:
train_df.isnull().sum()

In [ ]:
sns.countplot(data=train_df,x="Label")

In [ ]:
train_df["Date"] = pd.to_datetime(train_df["Date"])

In [ ]:
train_df["langth"] = train_df["reviews"].apply(len)

# Text Cleaning & Preprocessing

In [ ]:
def handleRepetitive(sentence):
    rx = re.compile(r'([^\W\d_])\1{2,}')
    return re.sub(r'[^\W\d_]+', lambda x: Word(rx.sub(r'\1\1', x.group())).correct() if rx.search(x.group()) else x.group(), sentence)

def replaceHomoglyphs(text):
    return unidecode(text)

def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '[URL]', text)

def remove_html(text):
    return BeautifulSoup(text, 'html.parser').get_text()

def remove_emails(text):
    canonical_email = re.sub(r'([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)','[EMAIL]', text)
    return canonical_email

def convertLowercase(text):
    return text.lower()

def remove_numbers(text):
    return re.sub(r'\d+', '[NUMBER]', text)

stop_words = set(nltk.corpus.stopwords.words("english"))
def removeStopWords(text):
    text = [word for word in text.split() if word not in stop_words]
    return ' '.join(text)

spetial_chars = string.punctuation
escaped_chars = [re.escape(c) for c in spetial_chars]
spetial_chars_regex = re.compile(f"({'|'.join(escaped_chars)})")

def remove_punctuation(text):
    return re.sub(spetial_chars_regex," ",text)

stemmer = nltk.stem.SnowballStemmer(language="english")

def stem(text):
    
    text = nltk.word_tokenize(text, language='english')
        
    text = [stemmer.stem(word) for word in text]
    
    return ' '.join(text)

In [ ]:
def clean_text(text):
    text = remove_punctuation(text)
    text = handleRepetitive(text)
    text = replaceHomoglyphs(text)
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_emails(text)
    text = remove_numbers(text)
    text = convertLowercase(text)
    text = removeStopWords(text)
    return text

In [ ]:
train_df["cleaned_reviews"] = train_df["reviews"].progress_apply(clean_text)

In [ ]:
train_df.to_csv(os.path.join("data/cleaned_data.csv"),index=False)

# Features Extraction

In [ ]:
train_df = pd.read_csv("./data/cleaned_data.csv")

In [ ]:
train_df['Label']=train_df['Label'].apply(lambda x: 1 if x=='Y' else 0)
train_df.head(3)

In [ ]:
def FeatureExtraction(df):
    reviewer_frequency = df.groupby('reviewer ID')['review ID'].count().reset_index()
    reviewer_frequency.columns = ['reviewer ID', 'review_frequency']

    ratios = df.groupby('reviewer ID').agg({
        'rating_Helpful': lambda x: sum(x) / len(x),
        'rating_Thanks': lambda x: sum(x) / len(x),
        'rating_LoveThis': lambda x: sum(x) / len(x),
        'review ID': 'count'
    }).reset_index()
    ratios.columns = ['reviewer ID', 'helpful_ratio', 'thanks_ratio', 'love_ratio', 'total_reviews']

    reviewer_behavior = pd.merge(reviewer_frequency, ratios, on='reviewer ID')

    # Calculate number of reviews per product
    product_reviews_count = df.groupby('product ID')['review ID'].count().reset_index()
    product_reviews_count.columns = ['product ID', 'reviews_count']
    
    rating_weights = {'rating_Helpful': 1, 'rating_Thanks': 1, 'rating_LoveThis': 1, 'rating_OhNo': -1}
    
    # Calculate overall rating for each product
    df['overall_rating'] = df.apply(lambda row: sum(row[rating] * rating_weights[rating] for rating in rating_weights.keys()), axis=1)
    
    # Calculate average rating per product
    product_avg_rating = df.groupby('product ID')['overall_rating'].mean().reset_index()
    product_avg_rating.columns = ['product ID', 'avg_rating']
    
    # Merge the two dataframes
    product_popularity = pd.merge(product_reviews_count, product_avg_rating, on='product ID')

    df = pd.merge(df, reviewer_behavior, on='reviewer ID')

    df = pd.merge(df, product_popularity, on='product ID')

    pol = lambda x: TextBlob(x).sentiment.polarity
    sub = lambda x: TextBlob(x).sentiment.subjectivity

    df['polarity'] = df['reviews'].progress_apply(pol)
    df['subjectivity'] = df['reviews'].progress_apply(sub)

    return df